# Εργαστήριο 4
*Όπως και στα προηγούμενα εργαστήρια, πριν ξεκινήσετε εκτελέστε τα επόμενα 2 κελιά*

In [ ]:
interp.configureCompiler(_.settings.processArguments(List("-Wconf:cat=deprecation:s"), true))
interp.load.module(os.Path(s"${System.getProperty("user.dir")}/resource/chisel_deps.sc"))

In [ ]:
import chisel3._
import chisel3.util._
import chiseltest._
import chiseltest.RawTester.test

## 4.1 Ακολουθιακά κυκλώματα με στοιχεία μνήμης
Στα προηγούμενα εργαστήρια ασχοληθήκαμε με **συνδυαστικά** κυκλώματα, λογικές δηλαδή συναρτήσεις που ανάλογα με τις εισόδους που δέχονται παράγουν τις αντίστοιχες εξόδους.

Στα επεξεργαστικά συστήματα όμως, η λειτουργία γίνεται **βηματικά σε φάσεις** που συγχρονίζονται από ένα η περισσότερα **σήματα ρολογιού** (clock). Μεταξύ των φάσεων πρέπει να κρατάμε τα προσωρινά αποτελέσματα σε "στοιχεία μνήμης" ή αλλιώς **καταχωρητές** (registers). Τα κυκλώματα που παράγουν αποτελέσματα με βάση όχι μόνο τις εισόδους αλλά και την προηγούμενη κατάσταση ονομάζονται **ακολουθιακά** κυκλώματα.

### 4.1.1 Τι είναι ένα στοιχείο μνήμης (καταχωρητής)
Είναι ένα κύκλωμα που "αποθηκεύει" την τιμή ενός αριθμού bits. Το πόσα bits θα έχει ένας καταχωρητής εξαρτάται από τον σχεδιασμό μας για να καλύπτει τις ανάγκες μας. Υποθέτοντας ότι τα bits είναι `N` στον αριθμό, ένας καταχωρητής έχει τις ακόλουθες εισόδους-εξόδους:

* **Είσοδος δεδομένων**, με εύρος `Ν` bits
* **Έξοδος δεδομένων**, με εύρος πάλι `Ν` bits
* **Ρολόι (clock)**, με εύρος 1 bit

και ενδεχομένως τα πρόσθετα σήματα εισόδου:

* **Είσοδος reset**, με εύρος 1 bit, για να αρχικοποιούμε τον καταχωρητή σε γνωστή **σταθερή** τιμή
* **Είσοδος enable**, με εύρος 1 bit, για να επιτρέπουμε ή όχι την εγγραφή νέων δεδομένων

### 4.1.2 Περιγραφή λειτουργίας καταχωρητή
1. Ο καταχωρητής ανά πάσα στιγμή έχει μια **αποθηκευμένη τιμή**, η οποία εμφανίζεται στην **έξοδο δεδομένων**.
2. Η **είσοδος δεδομένων** μπορεί να έχει οποιαδήποτε τιμή, αυτή **δεν επηρεάζει** την αποθηκευμένη τιμή (ούτε την έξοδο).
3. Κατά την **ανερχόμενη ακμή** του σήματος **clock** (όταν το clock από 0 γίνεται 1), η τιμή της **εισόδου δεδομένων** **εγγράφεται** στον καταχωρητή, διαγράφοντας τα παλιά δεδομένα.
4. Αμέσως μετά την αποθήκευση των νέων δεδομένων, αυτά εμφανίζονται και στην **έξοδο δεδομένων**.
5. Αν υπάρχει σήμα **enable**, αυτό θα πρέπει να είναι ενεργό (true) για να **επιτραπεί η εγγραφή**.
6. Αν υπάρχει **reset** και ενεργοποιηθεί, ο καταχωρητής παίρνει μια **προκαθορισμένη τιμή**, ανεξάρτητα από την προηγούμενη κατάστασή του. 

## 4.2 Οι καταχωρητές στην Chisel
Μπορούμε να ορίσουμε έναν καταχωρητή με **συγκεκριμένο εύρος** bits στην Chisel ως εξής:
~~~scala
val testReg = RegInit(0.U(8.W))
~~~
Το προηγούμενο ορίζει έναν καταχωρητή με όνομα `testReg` των **8 bits**. Το `0.U` στο όρισμα του `RegInit()` προσδιορίζει την **τιμή αρχικοποίησης** του καταχωρητή κατά την εκκίνηση του συστήματος.

Στη συνέχεια μπορούμε να συνδέσουμε την είσοδο του καταχωρητή έτσι ώστε να οδηγείται από κάποιο άλλο σήμα, π.χ. όπως στο επόμενο όπου ο καταχωρητής `testReg` δέχεται ως **είσοδο** το σήμα `io.in`:
~~~scala
testReg := io.in
~~~
Επίσης μπορούμε να συνδέσουμε ένα σήμα έτσι ώστε να παίρνει τιμή από την έξοδο του καταχωρητή, π.χ. στο επόμενο ο το σήμα `io.out` παίρνει τιμή από την **έξοδο** του καταχωρητή `testReg`:
~~~scala
io.out := testReg
~~~
**Προσοχή**: τα σήματα `clock` (και `reset`) **δεν τα γράφουμε εμείς**, τα παράγει αυτόματα η Chisel.

#### Παράδειγμα 
Δείτε το επόμενο παράδειγμα, όπου χρησιμοποιείται ένας καταχωρητής των 8 bits με τιμή αρχικοποίησης `255.U`. Η είσοδος `io.in` του module συνδέεται στην είσοδο του καταχωρητή και, αντίστοιχα, η έξοδος του καταχωρητή συνδέεται στην έξοδο `io.out` του module:

In [ ]:
class RegisterExample extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(8.W))
    val out = Output(UInt(8.W))
  })
  
  val testReg = RegInit(255.U(8.W))
  
  testReg := io.in
  io.out := testReg
}

Στο test που ακολουθεί, χρησιμοποιούμε τη μέθοδο `peek()` για να τυπώσουμε την τιμή που υπάρχει στην έξοδο `out` του module.

**Σημείωση:** παρατηρήστε πώς δίνουμε **έναν παλμό ρολογιού** με τη μέθοδο `step()` στο σήμα `clock`:
~~~scala
c.clock.step()
~~~

In [ ]:
try {
  test(new RegisterExample()) { c =>
    println(c.io.out.peek())   // τυπώνει την αρχική τιμή του καταχωρητή (δηλαδή 255)
    c.io.in.poke(123.U)        // αλλάζω την είσοδο στην τυχαία τιμή 123
    println(c.io.out.peek())   // όσο το clock δεν ενεργοποιείται, συνεχίζει να τυπώνει την αρχική τιμή (255)
    c.clock.step()             // δημιουργώ έναν παλμό ρολογιού
    println(c.io.out.peek())   // τώρα η είσοδος 123 θα αποθηκευτεί στον καταχωρητή και θα φανεί στην έξοδο 
  }
  println("✅ SUCCESS!!")
} catch {
  case e: chisel3.internal.ChiselException => 
    println(s"❌ Chisel Elaboration Failed: ${e.getMessage}")
  case e: Throwable => 
    println(s"❌ Chisel Testing Failed: ${e.toString}")
}

## 4.3 Υλοποίηση μετρητών με καταχωρητές
Τα κυκλώματα **μετρητή** (counters) είναι βασικό στοιχείο των επεξεργαστικών συστημάτων. Μπορούμε να υλοποιήσουμε μετρητές με τη βοήθεια καταχωρητών.

Στο επόμενο παράδειγμα χρησιμοποιται ένας καταχωρητής με αρχική τιμή 0. Η είσοδος του καταχωρητή συνδέεται στην έξοδό του, **αυξημένη κατά 1**.

Κατά τη λειτουργία του κυκλώματος, σε κάθε παλμό του ρολογιού αποθηκεύεται στον καταχωρητή μια νέα τιμή, η οποία είναι η παλιά + 1. Συνεπώς στην έξοδο του κυκλώματος θα δούμε μια ακολουθία τιμών αυξανόμενη κατά 1.

*Σημ.: Όταν η τιμή μέτρησης θα φτάσει στη μέγιστη (255 ή FF(hex) για το εύρος των 8 bits), τότε η επόμενη τιμή θα επιστρέψει στο 0 κ.ο.κ.*


In [ ]:
class CounterExample extends Module {
  val io = IO(new Bundle {
    val out = Output(UInt(8.W))
  })
  
  val testReg = RegInit(0.U(8.W))
  
  testReg := testReg + 1.U
  io.out := testReg
}

Εκτελέστε το πιο κάτω test για να δείτε την αυξανόμενη ακολουθία τιμών στην έξοδο:

In [ ]:
try {
  test(new CounterExample()) { c =>  
    println(c.io.out.peek())
    for (i <- 0 until 10) {
      c.clock.step()
      println(c.io.out.peek())
    }
  }
  println("✅ SUCCESS!!")
} catch {
  case e: chisel3.internal.ChiselException => 
    println(s"❌ Chisel Elaboration Failed: ${e.getMessage}")
  case e: Throwable => 
    println(s"❌ Chisel Testing Failed: ${e.toString}")
}    

## 4.4 Άσκηση 1
Υλοποιήστε κύκλωμα μετρητή με εύρος 8 bits, ο οποίος έχει μια είσοδο `offset` (με εύρος επίσης 8 bits). Συμπληρώστε στο επόμενο κελί τον κώδικα έτσι ώστε ο μετρητής να αυξάνεται **κατά +offset** αντί για +1. Ο μετρητής θα πρέπει να ξεκινάει από την τιμή 0.

In [ ]:
class OffsetCounter extends Module {
  val io = IO(new Bundle {
    val offset = Input(UInt(8.W))
    val out = Output(UInt(8.W))
  })
  
  // ..συμπληρώστε..
    
}

Συμπληρώστε τον κατάλληλο κώδικα έτσι ώστε να τυπώνεται η ακολουθία τιμών αυξανόμενη κατά offset = 10 (δηλ. 0, 10, 20, .. έως 100).

In [ ]:
try {
  test(new OffsetCounter()) { c =>
      
    // ..συμπληρώστε..
      
  }
  println("✅ SUCCESS!!")
} catch {
  case e: chisel3.internal.ChiselException => 
    println(s"❌ Chisel Elaboration Failed: ${e.getMessage}")
  case e: Throwable => 
    println(s"❌ Chisel Testing Failed: ${e.toString}")
}  

## 4.5 Άσκηση 2
Υλοποιήστε μετρητή με εύρος `Ν` bits και έξοδο `out` με εύρος ομοίως `N` bits, καθώς και με σήμα εισοδου `updown` με εύρος `1` bit. Όταν το `updown` είναι 1 ο μετρητής θα μετράει "προς τα πάνω" (+1). Στην αντίθετη περίπτωση θα μετράει "προς τα κάτω" (-1). Η αρχική τιμή του μετρητή θα είναι 0.

In [ ]:
class UpDownCounter(n:Int) extends Module {
  val io = IO(new Bundle {
    val updown = Input(UInt(1.W))
    val out = Output(UInt(n.W))
  })
  
  // ..συμπληρώστε..
    
}

Συμπληρώστε τον επόμενο κώδικα test για δημιουργήσετε ένα module `UpDownCounter` των 8 bits και να δείτε να τυπώνεται μια αυξανόμενη (0...10) και στη συνέχεια μια μειούμενη (10...0) ακολουθία τιμών στην έξοδο του μετρητή.

In [ ]:
try {
  test(new UpDownCounter(8)) { c =>
    
    // ..συμπληρώστε..
      
  }
  println("✅ SUCCESS!!")
} catch {
  case e: chisel3.internal.ChiselException => 
    println(s"❌ Chisel Elaboration Failed: ${e.getMessage}")
  case e: Throwable => 
    println(s"❌ Chisel Testing Failed: ${e.toString}")
}      

## Τελειώνοντας...
Μπορείτε να κατεβάσετε και να πάρετε μαζί σας το σημερινό notebook για να το έχετε στο αρχείο σας.

**Δεν ανήκει στα παραδοτέα του εργαστηρίου.**